# 39. RG3 — 가스 모델 + 미성형 모델, 검사 예산 나누기

| 역할 | 모델 |
|---|---|
| 가스 | RF, 원값 21개 + 새 변수 3개(전환 위치·평균 스크류 회전·최대 배압)의 직전 10샷 std (37번 반복 19) |
| 미성형 후보 | `under3_smote` (언더 3:1 + SMOTE) / `under10_adasyn` (언더 10:1 + ADASYN) / `balanced_rf` / `weight` (기준) |
| **미성형 중첩 선택** | 바깥 Run 하나씩 빼고 검증의 **학습 Run 안에서만** 다시 검증해 방식을 고름 → 결과 보고 고른 편향 없는 **정직한 추정** |

- 검사 예산 10%를 가스 몫 g / 미성형 몫 1−g로 나눔. g는 미리 정함: 1.0(가스만), 0.68(불량 수 비율 17:8), 0.5(반반)
- 시드 3개 평균·범위로 봄. 실행 시간: 시드당 수 분
- 판단 기준: **`중첩 선택` 행**이 목표(가스 ≥ 7 & 미성형 ≥ 4)를 넘는지. 고정 방식 행 중 최고는 참고용(낙관적)

**준비**: 20b 실행 (R6 제외 반영된 `outputs/final_model/rg3_scores.csv`), `pip install imbalanced-learn`

In [ ]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
import rg3_gas_short as G
R = G.run(seeds=(0, 1, 2))

## 1. 요약 (시드 3개 평균·최소·최대)

In [ ]:
display(R['summary'].round(3))

## 2. 중첩 선택에서 Run마다 고른 방식 (매번 같은 방식이 뽑히면 안정적)

In [ ]:
display(R['nested_log'].round(3))
print(R['nested_log']['고른 방식'].value_counts())

## 3. 시드별 상세

In [ ]:
display(R['by_seed'].round(3))

## 4. (선택) 순열검정 — 시간이 오래 걸림 (n=100이면 수십 분)
라벨 위치를 무작위로 옮기고 학습·선택 절차를 처음부터 반복 → 관측 포착 수가 우연을 넘는지

In [ ]:
# lab, X, cols = G.load()
# res = G.permutation_test(lab, X, method='중첩', gas_share=0.68, n=100)
# print(res)